In [ ]:
# ============================================================
# CELL 0: EXTRACT WHAT WE NEED FROM THE PREPROCESSING ZIP
# The input is one zip holding processed, processed_bg, and the manifest.
# We extract only processed_bg and the manifest, not the whole thing.
# ============================================================
import os, zipfile

ZIP = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip"
DEST = "/kaggle/working/data"

if not os.path.exists(f"{DEST}/manifest_v2.csv"):
    print("Extracting processed_bg and manifest (this takes a while)...")
    with zipfile.ZipFile(ZIP) as z:
        wanted = [n for n in z.namelist()
                  if n.startswith("processed_bg/") or n == "manifest_v2.csv"]
        print(f"Files to extract: {len(wanted):,}")
        z.extractall(DEST, members=wanted)
    print("Done!")
else:
    print("Already extracted, skipping")

print("Top level:", os.listdir(DEST))

In [ ]:
# ============================================================
# CELL 1: SETUP
# Load the tools and check the GPU is on
# ============================================================

# --- Built-in Python tools ---
import sys          # screen output
import json         # save results as text files
import time         # measure how long things take
import random       # random numbers
import logging      # timestamped messages saved to a file

# --- Data tools ---
import numpy as np          # arrays of numbers (images, scores)
import pandas as pd         # tables (the spreadsheet)
from PIL import Image       # open image files

# --- PyTorch and the pretrained ResNet ---
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader

# --- Scoring tools ---
from sklearn.metrics import (
    roc_auc_score,            # AUROC: how well DR is ranked above healthy
    roc_curve,                # every cutoff with its sensitivity and specificity
    average_precision_score,  # PR-AUC
    confusion_matrix,         # TP, FP, TN, FN counts
)
import matplotlib.pyplot as plt

# --- GPU check ---
# Without a GPU this would take days, so stop now if it's off
assert torch.cuda.is_available(), (
    "NO GPU. Settings > Accelerator > GPU T4 x2, then restart the session."
)
DEVICE = "cuda"
torch.backends.cudnn.benchmark = True  # pick the fastest convolution method

# --- Names and logging ---
RUN_NAME = "resnet18_all_layers_bce_balanced_bg"
OUT = "/kaggle/working"   # Kaggle's output folder, kept after the run

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),                  # screen
        logging.FileHandler(f"{OUT}/{RUN_NAME}.log"),       # file for mentors
    ],
    force=True,
)
log = logging.getLogger(RUN_NAME)

log.info("=" * 60)
log.info("RETINASENSE: ResNet-18, all layers trainable, plain BCE, balanced Ben Graham")
log.info("=" * 60)
log.info(f"GPU: {torch.cuda.get_device_name(0)} | PyTorch {torch.__version__}")

In [ ]:
# ============================================================
# CELL 2: SETTINGS
# Every number that could change the result lives here
# ============================================================

# --- Where the extracted Ben Graham data is ---
BASE = "/kaggle/working/data"
PROC = f"{BASE}/processed_bg"          # the Ben Graham preprocessed images
MANIFEST = f"{BASE}/manifest_v2.csv"   # the spreadsheet with labels and splits

# --- Training settings (shared config) ---
SEED = 42            # fixes randomness so runs repeat exactly
BATCH = 16           # images per step, smaller so ResNet fits in GPU memory
ACCUM = 2            # steps added together before updating -> effective batch 32
EPOCHS = 20          # maximum passes through the training data
PATIENCE = 5         # stop after this many epochs with no AUROC improvement
LR = 1e-4            # small on purpose: big steps damage pretrained layers
WD = 1e-4            # weight decay: small pull that keeps weights from growing huge
SENS_TARGET = 0.90   # threshold is chosen on validation to catch 90% of DR

# --- Set the seed everywhere ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

log.info("SETTINGS")
log.info(f"  Data:              {PROC}")
log.info(f"  Batch x accum:     {BATCH} x {ACCUM} = effective {BATCH * ACCUM}")
log.info(f"  Epochs / patience: {EPOCHS} / {PATIENCE}")
log.info(f"  Learning rate:     {LR} | weight decay {WD} | seed {SEED}")
log.info(f"  Sensitivity target for threshold: {SENS_TARGET:.0%}")

In [ ]:
# ============================================================
# CELL 3: LOAD AND CHECK THE DATA
# Read the spreadsheet, keep the usable images, and build loaders
# ============================================================

# --- Read the spreadsheet (one row per image) ---
df = pd.read_csv(MANIFEST)
log.info(f"Manifest rows: {len(df):,}")

# --- Keep only images that loaded correctly and were not excluded ---
ok = (
    df["load_ok"].astype(str).str.lower().eq("true")
    & df["excluded_reason"].isna()
)

# --- Split into train, val, test ---
# Balanced dataset: train is about 50% DR.
# The "unused" rows are NOT included, that is the imbalanced run.
splits = {}
for s in ["train", "val", "test"]:
    frame = df[(df["split"] == s) & ok].reset_index(drop=True)
    splits[s] = frame
    n_dr = int(frame["label"].sum())
    log.info(f"  {s:5s}: {len(frame):,} images | DR {n_dr:,} ({frame['label'].mean():.1%})")

# --- Check the files exist on disk (200 random ones per split) ---
for s, frame in splits.items():
    sample = frame["filename"].sample(min(200, len(frame)), random_state=SEED)
    missing = [f for f in sample if not os.path.exists(f"{PROC}/{s}/{f}")]
    if missing:
        raise FileNotFoundError(f"Missing in {s}: {missing[:3]}")
    log.info(f"  {s}: 200 random files checked, all found")

# --- Check the image size is 512 x 512 ---
first = splits["train"]["filename"].iloc[0]
size = Image.open(f"{PROC}/train/{first}").size
log.info(f"  Image size: {size} (expecting 512 x 512)")
assert size == (512, 512), f"Expected 512x512, got {size}"

# --- ImageNet normalisation numbers ---
# The pretrained ResNet was trained with these exact averages and spreads
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


class RetinaDataset(Dataset):
    """A recipe for loading one image and its label."""

    def __init__(self, frame, split_name):
        self.paths = [f"{PROC}/{split_name}/{f}" for f in frame["filename"]]
        self.labels = frame["label"].astype(np.float32).values

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = np.array(Image.open(self.paths[i]).convert("RGB"))  # height, width, colour
        x = torch.from_numpy(img).permute(2, 0, 1)                 # colour, height, width
        y = torch.tensor([self.labels[i]], dtype=torch.float32)    # label with shape (1,)
        return x, y


def make_loader(name, batch, shuffle):
    """Hands out batches. Only the training loader is shuffled."""
    return DataLoader(
        RetinaDataset(splits[name], name),
        batch_size=batch,
        shuffle=shuffle,
        num_workers=min(os.cpu_count() or 2, 4),  # helpers that load images in parallel
        pin_memory=True,
        persistent_workers=True,
    )


train_loader = make_loader("train", BATCH, shuffle=True)
val_loader = make_loader("val", 32, shuffle=False)
test_loader = make_loader("test", 32, shuffle=False)

# --- Look at one batch to confirm the shapes ---
xb, yb = next(iter(train_loader))
log.info(f"Batch check: images {tuple(xb.shape)} {xb.dtype}, labels {tuple(yb.shape)}")
log.info(f"  DR share in this batch: {yb.mean():.2f} (expecting about 0.50)")
log.info(f"  Batches per epoch: {len(train_loader)}")

In [ ]:
# ============================================================
# CELL 4: THE MODEL
# ResNet-18 with ImageNet weights. Every layer will learn.
# ============================================================

# --- Load ResNet-18 already trained on ImageNet ---
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# --- Replace the 1000-class output with one DR score ---
# ResNet-18's last layer takes 512 inputs
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(DEVICE)

# --- Make sure nothing is frozen: every parameter learns ---
for p in model.parameters():
    p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
log.info(f"Model: ResNet-18 (ImageNet weights) on GPU")
log.info(f"  Trainable parameters: {n_train:,} of {n_total:,}")

# --- Safety check: this run must train EVERYTHING ---
assert n_train == n_total, "Something is frozen, but this run should train all layers"
log.info("  Check passed: every layer is trainable")

In [ ]:
# ============================================================
# CELL 5: LOSS, OPTIMISER, MIXED PRECISION
# Plain BCE: no pos_weight, so missed DR and false alarms count the same
# ============================================================

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)

# Mixed precision: most maths in 16-bit, faster on the T4
scaler = torch.amp.GradScaler("cuda")

log.info("Loss: plain BCE (no class weight)")
log.info(f"Optimiser: Adam lr {LR}, weight decay {WD}")
log.info("Mixed precision: ON")

In [ ]:
# ============================================================
# CELL 6: TRAINING AND VALIDATION
# Train up to 20 epochs. Check validation AUROC after each one.
# Save the best. Stop after 5 epochs with no improvement.
# ============================================================


def to_input(x):
    """Scale 0-255 pixels to 0-1, then apply ImageNet normalisation."""
    x = x.to(DEVICE, non_blocking=True).float().div_(255.0)
    return (x - MEAN) / STD


@torch.no_grad()  # nothing is learned here, so don't track gradients
def predict(loader):
    """Score images without changing any weights."""
    model.eval()  # turn off dropout, use fixed batch-norm statistics
    scores, targets, total, n = [], [], 0.0, 0
    for x, y in loader:
        y = y.to(DEVICE)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(to_input(x))
        logits = logits.float()
        total += loss_fn(logits, y).item() * len(y)
        n += len(y)
        scores.append(torch.sigmoid(logits).cpu().numpy().ravel())
        targets.append(y.cpu().numpy().ravel())
    return np.concatenate(targets), np.concatenate(scores), total / n


history = []
best_auc = 0.0
bad_epochs = 0

log.info("=" * 60)
log.info("TRAINING STARTED")
log.info("=" * 60)

for epoch in range(1, EPOCHS + 1):
    # ---------- TRAINING: learn from every training image ----------
    model.train()  # turn on dropout and batch-norm learning
    t0 = time.time()
    run_loss, seen = 0.0, 0
    optimizer.zero_grad(set_to_none=True)

    for step, (x, y) in enumerate(train_loader, 1):
        y = y.to(DEVICE, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(to_input(x))         # step 1: predict
        loss = loss_fn(logits.float(), y)        # step 2: measure the error

        # step 3: backpropagation, divided by ACCUM so 2 small steps
        # add up to one normal-sized update
        scaler.scale(loss / ACCUM).backward()

        # step 4: update the weights every ACCUM steps
        if step % ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        run_loss += loss.item() * len(y)
        seen += len(y)
        if step == 1 or step % 200 == 0:
            log.info(f"  epoch {epoch} | batch {step}/{len(train_loader)} | "
                     f"running loss {run_loss / seen:.4f}")

    train_loss = run_loss / seen

    # ---------- VALIDATION: check progress on unseen images ----------
    y_val, s_val, val_loss = predict(val_loader)
    auc = roc_auc_score(y_val, s_val)
    minutes = (time.time() - t0) / 60

    # ---------- Is this the best epoch so far? ----------
    improved = auc > best_auc
    if improved:
        best_auc, bad_epochs = auc, 0
        torch.save(model.state_dict(), f"{OUT}/{RUN_NAME}_best.pt")
        status = "NEW BEST, saved"
    else:
        bad_epochs += 1
        status = f"no improvement ({bad_epochs}/{PATIENCE})"

    history.append({
        "epoch": epoch,
        "train_loss": round(train_loss, 5),
        "val_loss": round(val_loss, 5),
        "val_auroc": round(auc, 5),
        "minutes": round(minutes, 2),
    })

    log.info("-" * 60)
    log.info(f"EPOCH {epoch} COMPLETE in {minutes:.1f} min")
    log.info(f"  train loss {train_loss:.4f} | val loss {val_loss:.4f} | "
             f"val AUROC {auc:.4f} | best {best_auc:.4f} | {status}")
    log.info("-" * 60)

    if bad_epochs >= PATIENCE:
        log.info("EARLY STOPPING: validation AUROC stopped improving")
        break

log.info(f"TRAINING DONE: {len(history)} epochs, best val AUROC {best_auc:.4f}")

with open(f"{OUT}/{RUN_NAME}_history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ============================================================
# CELL 7: CHOOSE THE THRESHOLD
# Reload the best model, then find the cutoff that catches
# 90% of DR cases on the validation set
# ============================================================

model.load_state_dict(torch.load(f"{OUT}/{RUN_NAME}_best.pt", map_location=DEVICE))
log.info("Loaded best checkpoint")

y_val, s_val, _ = predict(val_loader)

fpr, tpr, thr = roc_curve(y_val, s_val)
i = np.where(tpr >= SENS_TARGET)[0][0]   # first cutoff reaching 90% sensitivity
threshold = float(thr[i])

log.info(f"THRESHOLD {threshold:.4f} | val sensitivity {tpr[i]:.3f} | "
         f"val specificity {1 - fpr[i]:.3f} | FROZEN for test")

In [ ]:
# ============================================================
# CELL 8: FINAL TEST
# Score the test set once with the frozen threshold
# DO NOT run this cell more than once
# ============================================================

y_test, s_test, test_loss = predict(test_loader)
pred = (s_test >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

results = {
    "run": RUN_NAME,
    "model": "resnet18",
    "trainable_layers": "all",
    "loss": "plain_bce",
    "preprocessing": "ben_graham",
    "threshold": float(threshold),
    "sensitivity": float(tp / (tp + fn)),
    "specificity": float(tn / (tn + fp)),
    "precision": float(tp / (tp + fp)) if (tp + fp) else 0.0,
    "auroc": float(roc_auc_score(y_test, s_test)),
    "pr_auc": float(average_precision_score(y_test, s_test)),
    "confusion": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}

log.info("RESULTS")
for k, v in results.items():
    log.info(f"  {k}: {v}")

with open(f"{OUT}/{RUN_NAME}_results.json", "w") as f:
    json.dump({"results": results, "history": history}, f, indent=2)
log.info("DONE. Do not rerun this cell.")

In [ ]:
# ============================================================
# CELL 9: TRAINING CURVES (for the mentors)
# ============================================================
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(h["epoch"], h["train_loss"], marker="o", label="train loss")
ax[0].plot(h["epoch"], h["val_loss"], marker="s", label="val loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Plain BCE loss, ResNet-18 all layers")
ax[0].legend()
ax[0].grid(True, alpha=0.3)

ax[1].plot(h["epoch"], h["val_auroc"], color="green", marker="o")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("AUROC")
ax[1].set_title("Validation AUROC")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{OUT}/{RUN_NAME}_curves.png", dpi=150)
plt.show()
log.info(f"Curves saved to {OUT}/{RUN_NAME}_curves.png")